# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the [FAIR² Mental Health Survey Dataset from Kilifi County, Kenya](https://sen.science/doi/10.71728/senscience.vcs2-05nj/) using the `mlcroissant` library, following the MLCommons Croissant standard.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Published: {metadata.datePublished}\nVersion: {metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs using the Croissant metadata.

In [ ]:
# List all record sets and their field @ids for inspection.

record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name','N/A')}")
    print(f"  description: {rs.get('description','')}")
    print(f"  fields:")
    for field in rs['fields']:
        print(f"    - @id: {field['@id']} | name: {field.get('name','N/A')} | dataType: {field.get('dataType','')} | description: {field.get('description','')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use record set and field `@id`s from the overview above.

_Below we select the main survey record set._

In [ ]:
# Replace these @ids with the ones you want to extract
# For this dataset, the main survey responses record set is typically the only one (as seen in the metadata)
# We'll automatically choose the first record set
main_record_set_id = record_sets[0]['@id']

# You can also get all records sets' @ids as a list
record_set_ids = [rs['@id'] for rs in record_sets]

dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)

# Data preview
print(f"Fields/columns in record set '{main_record_set_id}':")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below, we:
- Select the GAD-7 total score field (`gad7_total_score`) and group by gender,
- Filter responses with GAD-7 > 10 (indicative of moderate/severe anxiety),
- Normalize the scores for comparison across participants.

In [ ]:
# Define the field @id for GAD-7 total score and gender
# Use the actual @id names as found above (example placeholders used here)
df = dataframes[main_record_set_id]

# Find relevant columns. Typical field names (may need to visually inspect df.columns)
columns = df.columns.tolist()
print("Available columns:\n", columns)

# Try to select the most likely GAD-7 total score field
numeric_field = None
for col in columns:
    if 'gad7' in col.lower() and 'score' in col.lower():
        numeric_field = col
        break
if not numeric_field:
    raise ValueError('Could not find a GAD-7 score column.')

# Try to find 'gender' or similar for grouping
group_field = None
for col in columns:
    if 'gender' in col.lower() or 'sex' in col.lower():
        group_field = col
        break

print(f"Numeric field for analysis: {numeric_field}")
print(f"Grouping field: {group_field if group_field else 'N/A'}")

# Remove outliers: filter for GAD-7 > 10
threshold = 10
filtered_df = df[df[numeric_field] > threshold].copy()
print(f"\nFiltered records with {numeric_field} > {threshold}:")
print(filtered_df[[numeric_field]].head())

# Normalize the field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"\nNormalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Group by gender if available
if group_field and group_field in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index(name=f"mean_{numeric_field}")
    print(f"\nGrouped data by {group_field} (mean {numeric_field}):")
    print(grouped_df)
else:
    grouped_df = None

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the distribution of GAD-7 scores and, if available, the mean score by gender.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of GAD-7 total scores
plt.figure(figsize=(8,4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True, color='steelblue')
plt.title(f'Distribution of {numeric_field}')
plt.xlabel(numeric_field)
plt.ylabel('Count')
plt.show()

# Boxplot by gender if available
if group_field and group_field in df.columns:
    plt.figure(figsize=(6,4))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

## 6. Conclusion
In this notebook, we explored the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using `mlcroissant`. We:
- Loaded dataset metadata and inspected available record sets and fields (referenced by their `@id` values for reproducibility);
- Extracted and examined the survey responses, focusing on the GAD-7 anxiety scale;
- Performed basic filtering, normalization, grouping and visualization to gain insight into anxiety levels and demographics.

This notebook can be extended to analyze other record sets or fields, such as PHQ-9 or PSQ scores, or demographics such as age or education. For further research, refer to the MLCommons Croissant documentation and the dataset documentation at [https://sen.science/doi/10.71728/senscience.vcs2-05nj/](https://sen.science/doi/10.71728/senscience.vcs2-05nj/).